In [1]:
import os

repo_url = "https://github.com/chenzRG/Cancer-Multi-Omics-Benchmark.git"
repo_dir = "Cancer-Multi-Omics-Benchmark"

# Move to root if in src folder
if os.path.basename(os.getcwd()) == 'src':
    os.chdir("..")
    print("Moved to root directory")

# Clone or pull
if os.path.exists(repo_dir):
    print(f"Pulling latest changes...")
    !cd {repo_dir} && git pull
else:
    print(f"Cloning repository...")
    !git clone {repo_url}

Moved to root directory
Pulling latest changes...
Already up to date.


In [2]:
import pandas as pd
from huggingface_hub import hf_hub_download
import matplotlib.pyplot as plt

plt.style.use('dark_background')

repo_id = "AIBIC/MLOmics"
base = "Main_Dataset/Classification_datasets/GS-BRCA"

versions = {
    "Original": {
        "CNV":   "BRCA_CNV.csv",
        "Methy": "BRCA_Methy.csv",
        "mRNA":  "BRCA_mRNA.csv",
        "miRNA": "BRCA_miRNA.csv",
        "label": "BRCA_label_num.csv",
    },
    "Top": {
        "CNV":   "BRCA_CNV_top.csv",
        "Methy": "BRCA_Methy_top.csv",
        "mRNA":  "BRCA_mRNA_top.csv",
        "miRNA": "BRCA_miRNA_top.csv",
        "label": "BRCA_label_num.csv",
    },
    "Aligned": {
        "CNV":   "BRCA_CNV_aligned.csv",
        "Methy": "BRCA_Methy_aligned.csv",
        "mRNA":  "BRCA_mRNA_aligned.csv",
        "miRNA": "BRCA_miRNA_aligned.csv",
        "label": "BRCA_label_num.csv",
    },
}

all_dfs = {}  # all_dfs["Original"]["CNV"], etc.

for version, files in versions.items():
    print(f"\n{'='*40}")
    print(f"  {version}")
    print(f"{'='*40}")
    all_dfs[version] = {}

    for name, filename in files.items():

        if os.path.exists(f"data/{base}/{version}/{filename}"):
            path = f"data/{base}/{version}/{filename}"
        else:
            path = hf_hub_download(
                repo_id=repo_id,
                filename=f"{base}/{version}/{filename}",
                repo_type="dataset")
        
        df = pd.read_csv(path, index_col=0)

        if name == "label":
            df = df.index.to_series().reset_index(drop=True)
            all_dfs[version][name] = df
            print(f"  {name}: {df.shape} → value counts: {df.value_counts().to_dict()}")
        else:
            all_dfs[version][name] = df
            print(f"  {name}: {df.shape}")


  Original


  CNV: (19568, 671)
  Methy: (19049, 671)
  mRNA: (18206, 671)
  miRNA: (368, 671)
  label: (671,) → value counts: {0: 353, 2: 132, 4: 113, 1: 42, 3: 31}

  Top
  CNV: (5000, 671)
  Methy: (5000, 671)
  mRNA: (5000, 671)
  miRNA: (366, 671)
  label: (671,) → value counts: {0: 353, 2: 132, 4: 113, 1: 42, 3: 31}

  Aligned
  CNV: (11203, 671)
  Methy: (11189, 671)
  mRNA: (11343, 671)
  miRNA: (310, 671)
  label: (671,) → value counts: {0: 353, 2: 132, 4: 113, 1: 42, 3: 31}


In [28]:
import gseapy
gseapy.get_library_name(organism='Human')

['ARCHS4_Cell-lines',
 'ARCHS4_IDG_Coexp',
 'ARCHS4_Kinases_Coexp',
 'ARCHS4_TFs_Coexp',
 'ARCHS4_Tissues',
 'Achilles_fitness_decrease',
 'Achilles_fitness_increase',
 'Aging_Perturbations_from_GEO_down',
 'Aging_Perturbations_from_GEO_up',
 'Allen_Brain_Atlas_10x_scRNA_2021',
 'Allen_Brain_Atlas_down',
 'Allen_Brain_Atlas_up',
 'Azimuth_2023',
 'Azimuth_Cell_Types_2021',
 'BioCarta_2013',
 'BioCarta_2015',
 'BioCarta_2016',
 'BioPlanet_2019',
 'BioPlex_2017',
 'CCLE_Proteomics_2020',
 'CM4AI_U2OS_Protein_Localization_Assemblies',
 'COMPARTMENTS_Curated_2025',
 'COMPARTMENTS_Experimental_2025',
 'CORUM',
 'COVID-19_Related_Gene_Sets',
 'COVID-19_Related_Gene_Sets_2021',
 'Cancer_Cell_Line_Encyclopedia',
 'Carcinogenome',
 'CellMarker_2024',
 'CellMarker_Augmented_2021',
 'ChEA_2013',
 'ChEA_2015',
 'ChEA_2016',
 'ChEA_2022',
 'Chromosome_Location',
 'Chromosome_Location_hg19',
 'ClinVar_2019',
 'ClinVar_2025',
 'DGIdb_Drug_Targets_2024',
 'DSigDB',
 'Data_Acquisition_Method_Most_Popul

In [22]:
mirna = all_dfs['Original']['miRNA']
mlomics_mirnas = set(mirna.index)
print("miRNA index sample:\n", list(mlomics_mirnas)[:5], end="\n\n")

for lib_name in ['miRTarBase_2017', 'TargetScan_microRNA', 'TargetScan_microRNA_2017']:
    lib = gseapy.get_library(lib_name)
    lib_keys = set(lib.keys())  # ← keys, not values
    print(f"{lib_name:<25} {len(mlomics_mirnas & lib_keys):>3} / {len(mlomics_mirnas):3}", end="\t")
    print(f"DB keys:{str(list(lib.keys())[:2]):<40}", end='\t')
    print("DB values:", list(lib.values())[:2])

miRNA index sample:
 ['hsa.mir.19b.1', 'hsa.mir.664b', 'hsa.mir.203b', 'hsa.let.7e', 'hsa.mir.92b']

miRTarBase_2017             0 / 368	DB keys:['hsa-miR-507', 'hsa-miR-6723-5p']      	DB values: [['ZNF562', 'PTCD2', 'IMP4', 'CNBP', 'YIPF6', 'ANKEF1', 'CTNS', 'ACSL4', 'TMEM251', 'AKR7A2', 'FOXP2', 'SKI', 'UBE2Z', 'AAGAB', 'EOGT', 'COL9A2', 'HNRNPF', 'JCAD', 'PHIP', 'CALM2', 'KIAA0408', 'KANK4', 'FRK', 'CALM1', 'AKAP1', 'HCN4', 'ZNF154', 'USP6NL', 'ACSS3', 'CCNT2', 'SATB1', 'EI24', 'CERK', 'PARN', 'STRN3', 'BACH1', 'FNBP1L', 'BACH2', 'GTF2E2', 'PEX26', 'GPATCH8', 'ACTB', 'NWD1', 'PURB', 'TMEM245', 'EMC1', 'GPC4', 'ZNF703', 'ZNF701', 'JAZF1', 'EXPH5', 'AASDHPPT', 'POTED', 'SEMA6A', 'FAM9C', 'PAQR8', 'PAQR3', 'ST8SIA3', 'FST', 'LRRC40', 'DNAJC15', 'PARP15', 'PRRC2B', 'ACER3', 'RAET1E', 'REEP1', 'SMU1', 'GDNF', 'TADA3', 'CDC42EP3', 'ZNF813', 'ZNF410', 'RPL30', 'POGK', 'ZFP62', 'RAPH1', 'ARHGAP1', 'LRRC55', 'ETS1', 'ARPP19', 'ZNF25', 'SLC30A10', 'ZBTB5', 'WISP1', 'RBPJ', 'GUCD1', 'DYM', 'S

In [23]:
def normalize_mirna(name):
    # hsa.mir.9.1 → hsa-miR-9-1
    parts = name.replace('.', '-').split('-')
    if len(parts) >= 2:
        parts[1] = 'miR'
    return '-'.join(parts)

mirna_mapped = mirna.copy()
mirna_mapped.index = [normalize_mirna(m) for m in mirna.index]
mlomics_mirnas = set(mirna_mapped.index)
print("miRNA index sample:\n", list(mirna_mapped.index)[:5])


miRNA index sample:
 ['hsa-miR-7a-1', 'hsa-miR-7a-2', 'hsa-miR-7a-3', 'hsa-miR-7b', 'hsa-miR-7c']


In [24]:
for lib_name in ['miRTarBase_2017', 'TargetScan_microRNA', 'TargetScan_microRNA_2017']:
    lib = gseapy.get_library(lib_name)
    lib_keys = set(lib.keys())
    print(f"{lib_name:<25} {len(set(mirna_mapped.index) & lib_keys):>3} / {len(mirna_mapped.index):3}")


miRTarBase_2017            36 / 368
TargetScan_microRNA         0 / 368
TargetScan_microRNA_2017   43 / 368


https://gseapy.readthedocs.io/en/latest/introduction.html#gseapy-enrichr-module

In [25]:
# get target genes for matched miRNAs
lib = gseapy.get_library('miRTarBase_2017')
matched = set(mirna_mapped.index) & set(lib.keys())

target_genes = set()
for mirna in matched:
    target_genes.update(lib[mirna])

print(f"Matched miRNAs: {len(matched)}")
print(f"Total target genes: {len(target_genes)}")

# run enrichr on target genes against KEGG
result = gseapy.enrichr(
    gene_list=list(target_genes),
    gene_sets='KEGG_2021_Human',
    outdir=None,
)

Matched miRNAs: 36
Total target genes: 4615


In [31]:
result.results

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes
0,KEGG_2021_Human,Pathways in cancer,200/531,1.376185e-14,4.376269e-12,0,0,2.060266,6.575724e+01,CALML3;CALML4;ELK1;CRKL;FRAT2;RPS6KA5;AKT2;MYC...
1,KEGG_2021_Human,Proteoglycans in cancer,90/205,2.939074e-11,4.673128e-09,0,0,2.640980,6.404467e+01,ITGB1;CDKN1A;WNT2B;ITGB3;IHH;ELK1;FGF2;ACTB;AC...
2,KEGG_2021_Human,Breast cancer,70/147,5.403954e-11,5.728191e-09,0,0,3.061906,7.238746e+01,GSK3B;CDKN1A;WNT2B;PTEN;BRCA1;FGF1;BRCA2;FGF2;...
3,KEGG_2021_Human,Renal cell carcinoma,41/69,9.569502e-11,7.607754e-09,0,0,4.916274,1.134177e+02,CDKN1A;FH;CUL2;PRCC;SLC2A1;PDGFB;TGFA;PIK3R2;P...
4,KEGG_2021_Human,Cellular senescence,72/156,1.751081e-10,1.113687e-08,0,0,2.886890,6.485578e+01,CDKN1A;MRE11;PTEN;CALML3;CALML4;FOXM1;ETS1;PPP...
...,...,...,...,...,...,...,...,...,...,...
313,KEGG_2021_Human,Staphylococcus aureus infection,6/95,9.999935e-01,9.999956e-01,0,0,0.223734,1.456076e-06,C3;C1R;FPR1;FPR2;FCAR;MBL2
314,KEGG_2021_Human,Olfactory transduction,10/440,9.999936e-01,9.999956e-01,0,0,0.075525,4.812384e-07,SLC24A4;RGS2;GRK3;CAMK2D;PDE1B;CALML3;CALM3;CN...
315,KEGG_2021_Human,Neuroactive ligand-receptor interaction,41/341,9.999940e-01,9.999956e-01,0,0,0.450725,2.707782e-06,OPRD1;OXTR;ADCYAP1R1;GABRB1;NPFFR1;CHRM1;THRB;...
316,KEGG_2021_Human,Systemic lupus erythematosus,8/135,9.999955e-01,9.999956e-01,0,0,0.208625,9.492162e-07,CD86;C3;SNRPD1;C1R;SNRPD3;ACTN4;TRIM21;SNRPB


In [ ]:
# get target genes for matched miRNAs
lib = gseapy.get_library('TargetScan_microRNA_2017')
matched = set(mirna_mapped.index) & set(lib.keys())

target_genes = set()
for mirna in matched:
    target_genes.update(lib[mirna])

print(f"Matched miRNAs: {len(matched)}")
print(f"Total target genes: {len(target_genes)}")

# 2. run enrichr on target genes against KEGG
result = gseapy.enrichr(
    gene_list=list(target_genes),
    gene_sets='KEGG_2021_Human',
    outdir=None,
)

Matched miRNAs: 43
Total target genes: 12039


In [33]:
result.results

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes
0,KEGG_2021_Human,Pathways in cancer,397/531,5.952986e-13,1.904955e-10,0,0,1.991835,5.606960e+01,RB1;SPI1;ARAF;FZD10;ELK1;TFG;AKT2;AKT3;AKT1;PR...
1,KEGG_2021_Human,Axon guidance,149/182,2.260891e-10,2.688742e-08,0,0,3.010607,6.686587e+01,EPHB6;SEMA5A;SEMA5B;RGS3;DPYSL5;CFL2;DPYSL2;PL...
2,KEGG_2021_Human,Endocytosis,198/252,3.355245e-10,2.688742e-08,0,0,2.448470,5.341417e+01,VPS29;TFRC;WIPF1;ZFYVE9;WIPF2;WIPF3;ARPC5L;CBL...
3,KEGG_2021_Human,Proteoglycans in cancer,165/205,3.360927e-10,2.688742e-08,0,0,2.751737,6.002538e+01,HPSE2;ARAF;IHH;FZD10;ELK1;FGF2;TNF;ACTG1;IGF1R...
4,KEGG_2021_Human,MAPK signaling pathway,225/294,1.839232e-09,1.177109e-07,0,0,2.178329,4.381474e+01,ATF2;ARAF;ELK1;CRKL;ELK4;RPS6KA3;RPS6KA6;RPS6K...
...,...,...,...,...,...,...,...,...,...,...
315,KEGG_2021_Human,Ribosome,51/158,9.999948e-01,9.999960e-01,0,0,0.312270,1.629254e-06,RPL4;MRPS16;RPL32;RPL31;MRPS11;MRPS10;MRPL36;M...
316,KEGG_2021_Human,Systemic lupus erythematosus,30/135,9.999950e-01,9.999960e-01,0,0,0.186907,9.306843e-07,CD86;CD40;C1S;CD80;TNF;C8A;C5;GRIN2A;FCGR3A;C8...
317,KEGG_2021_Human,Oxidative phosphorylation,43/133,9.999950e-01,9.999960e-01,0,0,0.313487,1.554001e-06,ATP6V1A;NDUFB9;NDUFA11;UQCRB;NDUFB6;COX15;NDUF...
318,KEGG_2021_Human,Ribosome biogenesis in eukaryotes,38/108,9.999953e-01,9.999960e-01,0,0,0.356944,1.675565e-06,POP1;RPP30;WDR3;HEATR1;FCF1;NAT10;SPATA5;NXT2;...


In [48]:
# 1. build miRNA → target genes mapping
# lib = gseapy.get_library('miRTarBase_2017')
lib = gseapy.get_library('TargetScan_microRNA_2017')
matched = set(mirna_mapped.index) & set(lib.keys())

In [52]:
# 2. for each patient, build weighted gene scores
def build_patient_gene_scores(patient_mirna_expr, lib, matched):
    """weight target genes by miRNA expression for one patient"""
    gene_scores = {}
    for mirna in matched:
        expr = patient_mirna_expr.get(mirna, 0)
        for gene in lib[mirna]:
            gene_scores[gene] = gene_scores.get(gene, 0) + expr  # accumulate weights
    return pd.Series(gene_scores)

# 3. build gene score matrix (patients x genes)
patient_gene_scores = {}
for patient in mirna_mapped.columns:
    patient_expr = mirna_mapped[patient]
    patient_gene_scores[patient] = build_patient_gene_scores(patient_expr, lib, matched)

gene_score_matrix = pd.DataFrame(patient_gene_scores).fillna(0)  # genes x patients
print(f"Gene score matrix: {gene_score_matrix.shape}")  # expected: ~3000 genes x 671 patients

Gene score matrix: (12039, 671)


In [ ]:
gene_score_matrix.loc[gene_score_matrix.index[0]].head()
print(gene_score_matrix.shape) # shape (12039, 671) //TargetScan_microRNA_2017

(12039, 671)


In [79]:
# 4. run ssGSEA on weighted gene matrix
res = gseapy.ssgsea(
    data=gene_score_matrix,
    gene_sets='KEGG_2021_Human',
    outdir=None,
    sample_norm_method='rank',
    no_plot=True
)
mirna_pathway_scores = res.res2d.pivot(index='Name', columns='Term', values='NES')
# mirna_pathway_scores.columns = [f"miRNA_{c}" for c in mirna_pathway_scores.columns]
print(f"miRNA -> target genes -> pathway scores: {mirna_pathway_scores.shape}")  # 671 x 288 //TargetScan_microRNA_2017

miRNA -> target genes -> pathway scores: (671, 288)


In [80]:
display(all_dfs['Original']['miRNA'].iloc[:5,:5])
print(all_dfs['Original']['miRNA'].shape)

,TCGA.3C.AAAU.01,TCGA.3C.AALI.01,TCGA.3C.AALJ.01,TCGA.3C.AALK.01,TCGA.5L.AAT0.01
hsa.let.7a.1,0.068317,-0.301684,-0.150810,0.107831,0.395211
hsa.let.7a.2,0.068932,-0.318009,-0.122747,0.097594,0.412879
hsa.let.7a.3,0.073899,-0.301310,-0.126333,0.095545,0.418441
hsa.let.7b,0.524562,0.419859,-0.958939,0.615389,0.500594
hsa.let.7c,-1.656853,-0.715963,-0.971038,0.711952,0.426323


(368, 671)


In [81]:
display(mirna_pathway_scores.T.iloc[:5,:5])
print(mirna_pathway_scores.T.shape)

Name,TCGA.3C.AAAU.01,TCGA.3C.AALI.01,TCGA.3C.AALJ.01,TCGA.3C.AALK.01,TCGA.5L.AAT0.01
Term,,,,,
ABC transporters,-0.034433,0.222785,0.089604,0.158945,0.149530
AGE-RAGE signaling pathway in diabetic complications,0.198139,0.054318,0.102507,0.096153,-0.006928
AMPK signaling pathway,0.272981,0.141403,0.088929,0.053280,-0.012125
Acute myeloid leukemia,0.289507,0.105330,0.105666,-0.008227,-0.026570
Adherens junction,0.239619,0.000819,0.082747,0.055534,-0.009152


(288, 671)
